# 02 - Data Preparation

## Purpose 

After the EDA, this notebook implements the **Data Preparation** phase of the CRISP-DM, executing the transformation tasks deferred and noted from and in the EDA (notebook 01). The goal is to produce a clean, model-ready dataset, which output is consumed directly by the scorecard model in notebook 03.

## What his notebook produces

A prepared dataset saved as `application_prepared.parquet` (split into train and test), containing cleaned features, missingness indicators, and engineered variables, with multicollinear redundancies removed. 

## Scope

- **In scope:** sentinel handling, outlier treatment, missingness flags, feature engineering, and feature selection.
- **Out of scope:** Weight-of-Evidence binning and model fitting, part that belongs to the Modeling phase in the notebook 03.

## Key Principles 

Data preparation follows a single linear flow - **load raw, transform, save**, with all mutation confined to this notebook. Any transformation learned through the data is fitted solely on the **training split** and then applied to the test split. 

## Tasks carried forward from the EDA

The following items were flagged in the notebook 01 and are implemented here:

- `DAYS_EMPLOYED` sentinel (365243), Nan with an anomaly flag
- `AMT_INCOME_TOTAL` extreme outlier (117M), capping 
- `CODE_GENDER` invalid `XNA` rows, treatment
- `DAYS_*` negative integer encodings, positive units in years
- Missingness indicator flags for informative missing features
- Engineered ratios as credit-to-income, annuity-to-income, down-payment proxy
- Removal of multicollinear redundancies (housing triplets, duplicated features)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path 

# Anchor to the project root (using the samea as notebook 01)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw = ROOT / "data" / "raw"
processed = ROOT / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Project root:", ROOT)



Project root: c:\dev\credit-risk-pd-model


In [2]:
# Load the raw, untouched training data, not the EDAs mutated df. 
# Preparation must start from the original data so the transformation flow is 
# alway reproducible from the original data.

df = pd.read_parquet(processed / "application_train.parquet")

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Shape: 307,511 rows x 122 columns


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

## Train/test split

Before applying any data-driven transformation, the data is split into a training and a test sets. The split comes **first**, right before the data preparation, to prevent **data leakage**: Any transformation that *learns* from the data (imputation values, outlier caps, WoE bins) must be firred on the training set alone. If such values were computed over the full dataset before splitting, information from the test-set would bias the training process, producing an overly optimistic performance estimate that would not hold on genuinely unseein applicants (overfitting). 

The model is fitted on the training portion and evaluated on the test portion, which is treated as unseen future data. This measures whether the model generalises rather than memorising the training sample. 

Ideally the split would be **temporal** (training on older applicatnts, testing on newer) to mimic the real deployment process, where past data serves to predict future defaults. Unfortunately the Home Credit dataset lacks a usable application timestamp for a clean temporal split, so a **stratified random split** is used instead: random, but preserving the ~8% default rate in both sets so each remains representative. An 80/20 train/test ratio is applied. 

In [3]:
from sklearn.model_selection import train_test_split

# Separate target from features conceptually, but split the whole frame so
# feature columns and TARGET stay aligned per row (stratisfy). 

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["TARGET"],
    random_state=1,
)

print(f"Train: {train_df.shape[0]:,} rows  ({train_df['TARGET'].mean():.4f} default rate)")
print(f"Test:  {test_df.shape[0]:,} rows  ({test_df['TARGET'].mean():.4f} default rate)")

Train: 246,008 rows  (0.0807 default rate)
Test:  61,503 rows  (0.0807 default rate)


In [4]:
def clean_days_employed(frame):
    """Replace the DAYS_EMPLOYED sentinel with NaN and add an anomaly flag. Deterministic, no values learned from data, so it applies
    identically to train and test."""
    frame = frame.copy()
    sentinel = 365243
    frame["DAYS_EMPLOYED_ANOMALY"] = (frame["DAYS_EMPLOYED"] == sentinel).astype(int)
    frame["DAYS_EMPLOYED"] = frame["DAYS_EMPLOYED"].replace(sentinel, np.nan)
    return frame 

train_df = clean_days_employed(train_df)
test_df = clean_days_employed(test_df)

# Verifying on both sets

for name, frame in [("train", train_df), ("test", test_df)]:
    print(f"{name}: DAYS_EMPLOYED max = {frame['DAYS_EMPLOYED'].max():,.0f}, "
          f"NaN = {frame['DAYS_EMPLOYED'].isnull().sum():,}, "
          f"flag sum = {frame['DAYS_EMPLOYED_ANOMALY'].sum():,}")
          

train: DAYS_EMPLOYED max = 0, NaN = 44,226, flag sum = 44,226
test: DAYS_EMPLOYED max = 0, NaN = 11,148, flag sum = 11,148


In [5]:
days_cols = [c for c in train_df.columns if c.startswith("DAYS_")]
print(days_cols)

['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE', 'DAYS_EMPLOYED_ANOMALY']


In [6]:
def convert_days_to_years(frame):
    """Convert DAYS_* columns (negative days before application) to years.
    Deterministic. Excludes the DAYS_EMPLOYED_ANOMALY flag, which is a binary indicator, not a days measuremenet."""
    frame = frame.copy()
    days_cols = [
        c for c in frame.columns
        if c.startswith("DAYS_") and c != "DAYS_EMPLOYED_ANOMALY"
    ]
    for col in days_cols:
        years_col = col.replace("DAYS_", "YEARS_")
        frame[years_col] = -frame[col] / 365
    return frame, days_cols

train_df, converted = convert_days_to_years(train_df)
test_df, _ = convert_days_to_years(test_df)

print("Converted columns:", converted)
print()
print(train_df[["YEARS_BIRTH", "YEARS_EMPLOYED", "YEARS_REGISTRATION"]].describe().round(2))

Converted columns: ['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']

       YEARS_BIRTH  YEARS_EMPLOYED  YEARS_REGISTRATION
count    246008.00       201782.00           246008.00
mean         43.94            6.53               13.66
std          11.96            6.42                9.65
min          20.52           -0.00               -0.00
25%          34.02            2.10                5.50
50%          43.15            4.51               12.35
75%          53.93            8.69               20.48
max          69.12           49.07               65.04


**What I see:** Five `DAYS_*` columns were converted from negative days before application into new, positive `YEARS_*` columns, applied identically to the train and test sets. The `DAYS_EMPLOYED_ANOMALY` flag (a 0/1 indicator) was explicitly excluded from the conversion. The resulting ranges are then plausible: `YEARS_BIRTH` spans 20.5-69.1 years, consistent with the EDA, and `YEARS_EMPLOYED` spans 0-49.1 years, with 44,226 missing values in the training set that corresponds exactly to the nulled employment sentinel.

**Interpretation:** This is a unit conversion, not a rescaling. It makes the variables human-readableand keeps the eventual scorecard interpretable ("age in years" rather than "negative days"). Logical consistency checks passed: no applicant has more years employed or registered than years alive. 

**Decision:** The original `DAYS_*` columns are retained for now but are perfectly redundant with their `YEARS_*` counterparts. They will be dropped during feature selection. 

In [7]:
# Columns kept in the data but never used as model predictors.
# The dict values document WHY each is excluded. 
NON_FEATURE_COLS = {
    "SK_ID_CURR": "identifier",
    "TARGET": "label",
    "CODE_GENDER": "protected attribute, retained for fairness testing only",
}
def clean_code_gender(frame):
    """Map the invalid CODE_GENDER code 'XNA' to NaN. Determinisitc.
    CODE_GENDER is excluded from modelling and kept
    only for fairness evaluatioj in notebook 06."""
    frame = frame.copy()
    frame["CODE_GENDER"] = frame["CODE_GENDER"].replace("XNA", np.nan)
    return frame

train_df = clean_code_gender(train_df)
test_df = clean_code_gender(test_df)

for name, frame in [("train", train_df), ("test", test_df)]:
    print(name, frame["CODE_GENDER"].value_counts(dropna=False).to_dict())

train {'F': 162179, 'M': 83825, nan: 4}
test {'F': 40269, 'M': 21234}


### `CODE_GENDER`: excluded as a predictor

the 4 invalid `XNA` values were replaced with `NaN`. A dictionary, `NON_FEATURE_COLS`, records the columns that are retained in the data but never used as model predictors: the identifier `SK_ID_CURR`. the label `TARGET`, and `CODE_GENDER`.

Gender is **excluded from modelling** on legal and ethical grounds. It is **retained only for fairness testing** in notebook 06.

Excluding the attribute does not make the model gendert-neutral, because other features may be correlated with gender and act as proxies. Fairness is therefore evaluated on the model's output rather than assumed from exclusion. Note that the applicant population is about 66% female in both splits, so faiarness metrics will be reported per group. 

## Data-driven cleaning: income outlier cap

This section applies the first **data-driven** transformation, one that learns a value from the data. Following the fit/apply principle, the threshold is learned on the **training set only** and then applied to both sets. This prevents information from the test set leaking into the fitted transformation. 

`AMT_INCOME_TOTAL` is capped at 99.9th percentile of the training set: values above the threshold are set equal to it, and no rows are changed. Neither the WoE nor the tree-based challenger is sensitive to outliers, so the cap is a safeguard. It neutralises the implausible 117M value identified in notebook 01 before it can distort any magnitude calculation.  

In [8]:
INCOME_CAP_QUANTILE = 0.999
income_cap = train_df["AMT_INCOME_TOTAL"].quantile(INCOME_CAP_QUANTILE)
print(f"Income cap (train {INCOME_CAP_QUANTILE:.1%} percentile): {income_cap:,.0f}")

def cap_income(frame, cap):
    frame = frame.copy()
    frame["AMT_INCOME_TOTAL"] = frame["AMT_INCOME_TOTAL"].clip(upper=cap)
    return frame

# Affected rows

for name, frame in [("train", train_df), ("test", test_df)]:
    n = (frame["AMT_INCOME_TOTAL"] > income_cap).sum()
    print(f"{name}: {n} rows above the cap")

train_df = cap_income(train_df, income_cap)
test_df = cap_income(test_df, income_cap)

for name, frame in [("train", train_df), ("test", test_df)]:
    print(f"{name}: max income after cap = {frame['AMT_INCOME_TOTAL'].max():,.0f}")

Income cap (train 99.9% percentile): 900,000
train: 221 rows above the cap
test: 57 rows above the cap
train: max income after cap = 900,000
test: max income after cap = 900,000


**What I see:** The 99.9th percentile of `AMT_INCOME_TOTAL` in the training
set is 900,000, so this becomes our cap. 221 rows in the training set and 57
rows in the test set lie above it, about 0.09% in both cases. After capping,
the maximum income in both sets is exactly 900,000.

**Interpretation:** Slightly fewer rows than the expected 0.1% are capped,
because incomes are reported in round numbers and many applicants state
exactly 900,000. These sit at the cap, not above it. The share of capped rows
is also the same in train and test, which shows that a threshold learned only
on the training set fits the test set as well. All applicants are retained;
we only lower their reported income to the threshold. That considered, the
implausible 117M value from notebook 01 is neutralised, whichever set it
fell into.

**Decision:** The income ratios in the feature engineering section are
computed from the capped income. Since the cap was learned on the training
set only, the test set is treated exactly as new applicants would be.

## Missingness handling

**No imputation is applied.** The EDA showed that missingness carries signal: applicants without an external score default more often. Filling those gaps with a median would make thin-file applicants look average and erase that signal. Imputation is also unnecessary for both models: In the WoE scorecard, missing values receive their own bin with its own WoE. and LightGBM handles 'NaN' natively.

Instead, two row-level summary features are added:

- `HOUSING_MISSING_SHARE`: the share of the 47 housing columns ("_AVG", "_MODE", "_MEDI") that are missing. These columns tend to be missing together, and most of them will be removed during feature selection. This feature keeps the "housing data not collected" signal after they are gone, which is why this section comes before feature selection. 

- `EXT_SOURCE_MISSING_COUNT`: how many of the three external scores are missing (0-3), a combined proxy for a thin credit file. 

Both features are **deterministic**: counting missing values learns nothing from the data distributiion, so there are applied to the train and test sets directly. Wheter they caryy signal is checked on the training set only. 

In [9]:
HOUSING_SUFFIXES = ("_AVG", "_MODE", "_MEDI")
EXT_SOURCE_COLS = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

def add_missingness_features(frame):
    """Add row-level missingness summaries. Deterministic: counting missing
    values learns nothing from the data, so it applies to train and test
    identically. Individual columns keep their NaN values (no imputation)."""
    frame = frame.copy()
    housing_cols = [c for c in frame.columns if c.endswith(HOUSING_SUFFIXES)]
    frame["HOUSING_MISSING_SHARE"] = frame[housing_cols].isnull().mean(axis=1)
    frame["EXT_SOURCE_MISSING_COUNT"] = frame[EXT_SOURCE_COLS].isnull().sum(axis=1)
    return frame, housing_cols

train_df, housing_cols = add_missingness_features(train_df)
test_df, _ = add_missingness_features(test_df)

print(f"Housing columns used: {len(housing_cols)}")
print(housing_cols)

Housing columns used: 47
['APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'TOTALAREA_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


In [10]:
# Signal check on the TRAINING set only.
housing_group = pd.cut(
    train_df["HOUSING_MISSING_SHARE"],
    bins=[-0.01, 0, 0.99, 1],
    labels=["none missing", "partly missing", "all missing"],
)
print(train_df.groupby(housing_group, observed=True)["TARGET"].agg(["size", "mean"]).round(4))
print()
print(train_df.groupby("EXT_SOURCE_MISSING_COUNT")["TARGET"].agg(["size", "mean"]).round(4))

                         size    mean
HOUSING_MISSING_SHARE                
none missing            62759  0.0689
partly missing          66699  0.0703
all missing            116550  0.0931

                            size    mean
EXT_SOURCE_MISSING_COUNT                
0                          87681  0.0728
1                         128760  0.0817
2                          29422  0.0998
3                            145  0.0897


**What I see:** The suffix pattern selects exactly the 47 housing columns.
In the training set, the default rate is nearly identical whether housing
data is fully present (6.89%) or partly missing (7.03%), but rises to 9.31%
when all housing data is missing, which applies to 47% of applicants. For
the external scores, the default rate rises steadily with the number of missing scores: 7.28% (none), 8.17% (one), 9.98% (two). The group with all three missing contains only 145 applicants (8.97%).

**Interpretation:** Housing missingness acts as an all-or-nothing signal rather
than a gradient, consistent with the block being collected or not collected as 
a whole. The external-score count confirms the thin-file pattern from the EDA as a combined measure. The apparent dip at three missing scores is not interpretable: with n = 145 the standard error is about +-2.4 percentage points, so it cannot be distinguished from the two-missing group. Both effects are moderate compared with the external scores themselves.

**Decision:** Both features are retained. Their predictive value will be quantified by Information Value in notebook 03, where WoE binning will
capture the all-missing split for housing and merge the sparse three-missing
group. Individual columns keep their `NaN` values, as decided above.

## Feature engineering

**Feature engineering** derives new variables that capture risk better than
the raw data. Raw amounts can be misleading on their own: a €300k loan is
harmless for a high earner and unaffordable for a low earner. What matters
in credit underwriting is the relationship between amounts, so four ratios
are added, each relating two amounts to each other:

| Feature | Formula | Credit meaning |
|---|---|---|
| `CREDIT_INCOME_RATIO` | `AMT_CREDIT / AMT_INCOME_TOTAL` | Leverage: loan size relative to annual income |
| `ANNUITY_INCOME_RATIO` | `AMT_ANNUITY / AMT_INCOME_TOTAL` | Payment burden: share of income spent on repayments (debt-service-to-income) |
| `ANNUITY_CREDIT_RATIO` | `AMT_ANNUITY / AMT_CREDIT` | Repayment speed: a higher value implies a shorter loan term |
| `CREDIT_GOODS_RATIO` | `AMT_CREDIT / AMT_GOODS_PRICE` | Loan-to-value proxy: amount financed relative to the price of the goods |

`ANNUITY_CREDIT_RATIO` was added beyond the EDA plan because the implied
loan term is not captured by the other ratios. The loan-to-value proxy is a
ratio rather than a difference (`AMT_GOODS_PRICE − AMT_CREDIT`) because a
ratio is scale-free and comparable across loan sizes.

A log transform of income is **not** applied. WoE binning and tree splits
both depend only on the order of values, which a monotonic transformation
such as the logarithm does not change. Both models would therefore produce
identical results with or without it.

All four ratios are **deterministic**: each value uses only that applicant's
own amounts and learns nothing from the data, so the features are applied
to the train and test sets directly without risk of leakage. A zero
denominator is treated as missing to avoid infinite values.

In [11]:
def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)

def add_ratio_features(frame):
    frame = frame.copy()
    frame["CREDIT_INCOME_RATIO"] = safe_divide(frame["AMT_CREDIT"], frame["AMT_INCOME_TOTAL"])
    frame["ANNUITY_INCOME_RATIO"] = safe_divide(frame["AMT_ANNUITY"], frame["AMT_INCOME_TOTAL"])
    frame["ANNUITY_CREDIT_RATIO"] = safe_divide(frame["AMT_ANNUITY"], frame["AMT_CREDIT"])
    frame["CREDIT_GOODS_RATIO"] = safe_divide(frame["AMT_CREDIT"], frame["AMT_GOODS_PRICE"])
    return frame

RATIO_COLS = [
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "ANNUITY_CREDIT_RATIO",
    "CREDIT_GOODS_RATIO",
]

train_df = add_ratio_features(train_df)
test_df = add_ratio_features(test_df)

print(train_df[RATIO_COLS].describe().T.round(3))
print("\nMissing values (train):")
print(train_df[RATIO_COLS].isnull().sum())
print("\nInfinite values (train):")
print(np.isinf(train_df[RATIO_COLS]).sum())

                         count   mean    std    min    25%    50%    75%     max
CREDIT_INCOME_RATIO   246008.0  3.961  2.690  0.083  2.022  3.274  5.164  84.737
ANNUITY_INCOME_RATIO  245998.0  0.181  0.094  0.008  0.115  0.163  0.229   1.876
ANNUITY_CREDIT_RATIO  245998.0  0.054  0.022  0.022  0.037  0.050  0.064   0.124
CREDIT_GOODS_RATIO    245795.0  1.123  0.124  0.150  1.000  1.119  1.198   6.000

Missing values (train):
CREDIT_INCOME_RATIO       0
ANNUITY_INCOME_RATIO     10
ANNUITY_CREDIT_RATIO     10
CREDIT_GOODS_RATIO      213
dtype: int64

Infinite values (train):
CREDIT_INCOME_RATIO     0
ANNUITY_INCOME_RATIO    0
ANNUITY_CREDIT_RATIO    0
CREDIT_GOODS_RATIO      0
dtype: int64


In [12]:
for col in RATIO_COLS:
    deciles = pd.qcut(train_df[col], q=10, duplicates="drop")
    rates = train_df.groupby(deciles, observed=True)["TARGET"].mean()
    print(f"{col}: {rates.round(3).tolist()}")

CREDIT_INCOME_RATIO: [0.068, 0.077, 0.081, 0.089, 0.087, 0.093, 0.087, 0.08, 0.075, 0.07]
ANNUITY_INCOME_RATIO: [0.072, 0.072, 0.078, 0.078, 0.079, 0.084, 0.085, 0.088, 0.088, 0.083]
ANNUITY_CREDIT_RATIO: [0.059, 0.071, 0.074, 0.051, 0.097, 0.092, 0.099, 0.099, 0.112, 0.049]
CREDIT_GOODS_RATIO: [0.067, 0.048, 0.068, 0.067, 0.087, 0.1, 0.099, 0.129]


**What I see:** All four ratios are computed without infinite values. Only
a few values are missing, coming from missing inputs: 10 for the two annuity
ratios (`AMT_ANNUITY`) and 213 for `CREDIT_GOODS_RATIO` (`AMT_GOODS_PRICE`).
The typical applicant spends about 16% of income on repayments, repays in
about 20 instalments, and finances about 12% more than the goods price.
Across the deciles of the training set, the default rate follows a different
shape for each feature:
- `CREDIT_GOODS_RATIO` rises overall, from 6.7% to 12.9%, with a dip in the
  second bin (8 bins instead of 10, because many loans have a ratio of
  exactly 1.0).
- `ANNUITY_CREDIT_RATIO` is irregular, ranging from 4.9% to 11.2%, with
  sharp drops at deciles 3 and 9.
- `CREDIT_INCOME_RATIO` forms a hump, peaking at 9.3% in the middle deciles.
- `ANNUITY_INCOME_RATIO` mostly rises, from 7.2% to 8.8%.

**Interpretation:** The ratios describing how a loan is structured carry
clearly more signal than the affordability ratios. That considered, how a
loan is structured predicts default better than its size relative to
income. The rising `CREDIT_GOODS_RATIO` fits the loan-to-value logic:
financing more relative to the goods price means higher risk. The irregular
pattern of `ANNUITY_CREDIT_RATIO` is not noise: with about 24,600 applicants
per decile the standard error is about 0.17 percentage points, while the
jumps are about 4 points. A possible explanation, not verified, is that loans
come in standard terms, so deciles align with product types. The hump in
`CREDIT_INCOME_RATIO` reappears on the training set alone, which supports
the EDA finding. The income cap raised the minimum `CREDIT_INCOME_RATIO` from
0.005 to 0.083, but did not change the decile patterns, since deciles only
depend on the order of values.

**Decision:** All four ratios are retained. In notebook 03, each gets its
own binning trend: peak for `CREDIT_INCOME_RATIO`, ascending for
`ANNUITY_INCOME_RATIO` and `CREDIT_GOODS_RATIO`, and unconstrained for
`ANNUITY_CREDIT_RATIO`. Their final value will be measured by Information
Value. The extreme maximum of `CREDIT_INCOME_RATIO` (84.7) needs no further
treatment, because WoE binning places it in the top bin.

## Feature selection

The EDA identified 113 features pairs with a correlation above 0.8. These pairs represent two different kinds of redundancy:

-**Definitional redundancy:** the same quantity appears more than once, for example a variable and its rescaled version, or three summaries of the same building measurement. Keeping both adds no information, and the decision does not require looking at the target.

-**Statistical redundancy:** different concepts that happen to move together, such as `CNT_CHILDREN` and `CNT_FAM_MEMBERS`. Choosing between them requires knowing which is more predictive, which is measured by Information Value during WoE binning. 

This section therefore removes **only definitional redundancy**. Statistical redundancy is resolved in notebook 03, where the feature with the higher Information Value is retained from each correlated pair.

| Removed | Kept | Reason |
|---|---|---|
| 5 original `DAYS_*` columns | `YEARS_*` | Same quantity, rescaled |
| 28 housing `_MODE` and `_MEDI` variants | `_AVG` columns and the 5 `_MODE`-only columns | Three summaries of one measurement (r ≈ 0.99) |
| `AMT_GOODS_PRICE` | `AMT_CREDIT`, `CREDIT_GOODS_RATIO` | Recoverable as credit / ratio |
| `FLAG_EMP_PHONE` | `DAYS_EMPLOYED_ANOMALY` | Same employment information, inversely coded; identical except for 9 of 246,008 training rows |

Inside each housing triplet the choice of `_AVG` is arbitrary, since the
three variants are almost perfectly correlated. `AMT_CREDIT` is kept over
`AMT_GOODS_PRICE` because it represents the exposure and is needed to
approximate the exposure at default (EAD) planned for the IFRS 9 appendix.

The columns in `NON_FEATURE_COLS` (the identifier, the target and
`CODE_GENDER`) are **not** dropped. They remain in the saved data and are
excluded only at modelling time.

In [13]:
# How often each combination of the two values occurs. 
pd.crosstab(train_df["FLAG_EMP_PHONE"], train_df["DAYS_EMPLOYED_ANOMALY"])

DAYS_EMPLOYED_ANOMALY,0,1
FLAG_EMP_PHONE,,
0,9,44226
1,201773,0


In [14]:
#Going for deninitional redundancy here. 
days_originals = [
    "DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE",
]

#Housing keeping _AVG for each measurement, dropping _MODE and _MEDI twins. 

avg_cols = [c for c in train_df.columns if c.endswith("_AVG")]
# For each average column, there is a _MODE and a _MEDI. 
housing_twins = [c.replace("_AVG", suffix) for c in avg_cols for suffix in ("_MODE", "_MEDI")]

DROP_COLS = days_originals + housing_twins + ["AMT_GOODS_PRICE", "FLAG_EMP_PHONE"]

n_before = train_df.shape[1]
train_df = train_df.drop(columns=DROP_COLS)
test_df = test_df.drop(columns=DROP_COLS)

print(f"Dropped {len(DROP_COLS)} columns: {n_before} -> {train_df.shape[1]}")
print(f"Housing twins dropped: {len(housing_twins)}")
assert list(train_df.columns) == list(test_df.columns), "Train and test columns differ"
print("Train and test have identical columns.")

Dropped 35 columns: 134 -> 99
Housing twins dropped: 28
Train and test have identical columns.


**What I see:** [35] columns were removed, reducing the dataset from [134] to [99] columns, with identical columns in the train and test sets. The removed columns aare the 5 original `DAYS_*` columns, the 28 housing `_MODE` and `_MEDI` twins, `AMT_GOODS_PRICE` and `FLAG_EMP_PHONE`. The crosstab executed two cells above shows that `FLAG_EMP_PHONE` is inversely correlated to `DAYS_EMPLOYED_ANOMALY`: all the 44,226 applicants flagges as not employed have no employer phone. The two columns disagree for only 9 of 246,008 training rows (0.004%).

**Interpretation:** All removed columns duplicated information already present in a retained  column. The correlation of 1.000 reported in the EDA was rounded and hid the 9 exceptions, a reason to always check near-perfect correlations. The 9 rows are most likely inconsistent records and carry no measurable predictive information. 

**Decision:** Only definitional redundancy has been removed. Correlations between distinct concepts reimain in the data and will be resolved in notebook 03 using Information Values. 

## Saving the prepared data

The prepared train and tests sets are now saved as separate Parquet files in `data/processed/`, which serve as the input for notebook 03. Before saaving, a final set of integrity checks confirms that both sets have identical columns, that no rows were lost, that no infinite values remain, and that the columns required lated (`NON_FEATURE_COLS`) are present.

The parameters leearned or decided in this notebook are saved to `configs/preparation_params.json`: the split settings, the income cap fitted on the training set, the list of non-feature columns and the dropped columns. Any new data must then be transformed with exactly these values, not with estimaations from the new data. 

In [15]:
import json

# Integrity checks

assert list(train_df.columns) == list(test_df.columns)
assert len(train_df) + len(test_df) == len(df)
assert not np.isinf(train_df.select_dtypes("number")).any().any()
assert not np.isinf(test_df.select_dtypes("number")).any().any()

for col in NON_FEATURE_COLS:
    assert col in train_df.columns, f"Required column missing: {col}"

# Save the prepared datasets. 

train_path = processed / "train_prepared.parquet"
test_path = processed / "test_prepared.parquet"
train_df.to_parquet(train_path, index=False)
test_df.to_parquet(test_path, index=False)

# Save learned parameters and decisions.

params = {
    "split": {"test_size": 0.20, "random_state": 42, "stratify": "TARGET"},
    "income_cap": {"quantile": INCOME_CAP_QUANTILE, "value": float(income_cap)},
    "non_feature_cols": NON_FEATURE_COLS,
    "dropped_cols" : DROP_COLS,
}
configs = ROOT / "configs"
configs.mkdir(exist_ok=True)
with open(configs / "preparation_params.json", "w") as f:
    json.dump(params, f, indent=2)

assert pd.read_parquet(train_path).shape == train_df.shape
assert pd.read_parquet(test_path).shape == test_df.shape

print(f"Saved {train_path.name}: {train_df.shape[0]:,} rows x {train_df.shape[1]} columns")
print(f"Saved {test_path.name}: {test_df.shape[0]:,} rows x {test_df.shape[1]} columns")
print(f"Saved configs/preparation_params.json")

Saved train_prepared.parquet: 246,008 rows x 99 columns
Saved test_prepared.parquet: 61,503 rows x 99 columns
Saved configs/preparation_params.json


**What I see:** The prepared training set (246,008 rows × 99 columns) and test set (61,503 rows × 99 columns) were saved to `data/processed/`. All integrity checks passed, identical columns in both sets, no rows lost relative to the 307,511 raw applications, no infinite values, and all `NON_FEATURE_COLS`present. Realoading the files reproduces the same shapes.

**Decision:** Notebook 03 starts from these two files and does not aaccess the raw data. The fitted income cap, the split settingm the non-feature columns and the droped columns are stored in `configs/preparaation_params.json` for reuse. 

## Summary and handoff to notebook 03

This notebook transformed the raw application data into a model ready train and test sets, fitting all data-driven transformations on the training set only.

**Notebook 03 receives:**
- `train_prepared.parquet`and `test_prepared.parquet`, with the original 80/20 stratified split preserved.
- `configs/preparaation_params.json`, containing the columns to exclude from modelling and the fitted parameters.

**State of the data:**
- Sentinel values replaced, `DAYS_*` converted to years, invalid `CODE_GENDER`values set to missing, and income capped at 900,000.
- Missing values are deliberately **not imputed**. WoE binning assigns them their own bin.
- Added features: `DAYS_EMPLOYED_ANOMALY`, `HOUSING_MISSING_SHARE`, `EXT_SOURCE_MISSING_COUNT` and four affordability and loan-structure ratios.
- Only definitional redundancy has been removed. 

**Tasks for notebook 03:**
- WoE binning with a trend per feature, as established in Notebook 01 and 02: descending for `EXT_SOURCE_*` and age, peak for `CREDIT_INCOME_RATIO`, ascending for `ANNUITY_INCOME_RATIO` and `CREDITS_GOODS_RATIO`, unconstrained for `ANNUITY_CREDIT_RATIO`. 
- Feature selection by Information Value, including the resolution of the remaining correlated pairs by keeping the one with the higher Information Value between them.
- Logistic regression on the WoE-transformed features and scorecard scaling.
- The test set stays untouched until final evaluation.  